<a href="https://colab.research.google.com/github/JuliaMcPhillips/ds2002-fa26/blob/main/Copy_of_2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [38]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _line item (one item and its quantity from one order). Orders with multiple items get split into multiple rows, and I keep the order number and vendor ID on each row so I can still tell where it came from._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [39]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order','vendor_id'])
assert len(lines) == 4 and lines['qty'].sum() == 7
print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [40]:
vendors = pd.json_normalize(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
un = joined[joined['_merge'] == 'left_only']
print(joined)
print("Unmatched vendor(s):", un['vendor_id'].unique().tolist())
print("Units attached:", un['qty'].sum())
#Vendor V-07 did not match, and 3 units (the Hot Dogs) are attached to it.

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
Unmatched vendor(s): ['V-07']
Units attached: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [41]:
# Keep the unmatched vendor as its own labeled row instead of dropping it.
# groupby silently drops NaN keys, which would lose 3 of the 7 units with no warning.
# Labeling it with its vendor_id keeps the total at 7 and makes the gap visible.
joined['name'] = joined['name'].fillna('UNKNOWN (' + joined['vendor_id'] + ')')
q3 = joined.groupby('name')['qty'].sum().sort_values(ascending=False)
print(q3); print("total:", q3.sum())

name
Hoos Burgers      3
UNKNOWN (V-07)    3
Cav Merch         1
Name: qty, dtype: int64
total: 7


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [42]:
zone_default = joined.groupby('zone')['qty'].sum()            # default: NaN zone dropped
zone_all = joined.groupby('zone', dropna=False)['qty'].sum()  # keeps the NaN group
print(zone_default); print("total:", zone_default.sum())
print(zone_all); print("total:", zone_all.sum())
print("Top zone:", zone_default.idxmax())
# Zone A sold the most, with 4 items.
# With the default groupby, my zone total is 4, not 7.
# The missing 3 units are the V-07 Hot Dogs: V-07 has no vendor record,
# so its zone is NaN, and groupby drops NaN keys.
# With dropna=False they show up as their own unknown-zone group
# and the total is 7 again.

zone
A    4
Name: qty, dtype: int64
total: 4
zone
A      4
NaN    3
Name: qty, dtype: int64
total: 7
Top zone: A


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [43]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

rows = []
for o in ragged_json:
    for ln in (o.get('lines') or [None]):      # order with no 'lines' still gets one row
        rows.append({'order': o['order'], 'vendor_id': o['vendor_id'],
                     'item': None if ln is None else ln.get('item'),
                     'qty':  None if ln is None else ln.get('qty')})
r = pd.DataFrame(rows)
r['qty'] = pd.to_numeric(r['qty']).astype('Int64')   # nullable int: missing = <NA>, zero stays 0
print(r); print(r.dtypes)
print("sum of known qty:", r['qty'].sum(), "| missing:", r['qty'].isna().sum())

   order vendor_id   item   qty
0      4      V-01   Soda     2
1      5      V-10   None  <NA>
2      6      V-01  Fries  <NA>
order         int64
vendor_id    object
item         object
qty           Int64
dtype: object
sum of known qty: 2 | missing: 2


### Q6 — Validate

In [44]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_a)"One row per line item" is the useful shape here because the line item is the unit that actually has an item and a quantity. Order is just a container. With that grain, questions like total units per item, per vendor or per zone become a simple groupby and sum. With one row per order, the lines would be a list stuck inside a cell, and you'd have to unpack it again before you could count or aggregate anything. It also makes the join safe, since each line item has only one vendor, so adding the vendor info doesn't change the number of rows.

b) A missing quantity means "we don't know how many were sold", while zero means "we know none were sold". Treating them as the same value, for example with fillna(0), turns unknowns into facts. Totals and averages then come out understated, and inventory or revenue numbers look more certain than they are. A "Fries" order with an unrecorded quantity would look like Fries sold zero. Keeping the missing quantity as `<NA>` (using a nullable Int64 column) means sums skip the unknowns, and I can still count, report and chase the bad records._